# Task annotator

Hacky notebook to quickly generate manual navigation tasks for training/evaluation

### Insert the path of an occupancy gridmap png file

In [ ]:
from cfg.CFG import SCENE_USD_PATH

MAP_PATH_ = str(SCENE_USD_PATH).replace(".usda", "_map.png")  # full/relative path to the map file, the resulting task yaml file will be created in the same directory
 

In [ ]:
import pyastar2d
import scipy.ndimage as sp
import numpy as np

def get_astar_path(img, start, goal) -> np.ndarray:
    binary_img = img == 0 # 1= occupied, 0=free/unknown
    dist_transform = sp.distance_transform_edt(~binary_img)
    costmap = np.clip(np.max(dist_transform) - (dist_transform * 3), 0, 255).astype(np.float32)
    max_value = costmap.max()
    costmap[costmap >= max_value-1] = float('inf')
    # plt.imshow(costmap, cmap='gray')
    # plt.show()
    return pyastar2d.astar_path((costmap+1), start, goal, allow_diagonal=True)

In [ ]:
# If you run this cell standalone, ensure the interactive widget backend is enabled.
# (VS Code/Jupyter sometimes loses the backend state after a restart.)

from IPython import get_ipython
ip = get_ipython()
if ip is not None:
    ip.run_line_magic("matplotlib", "widget")

import os
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import FancyArrowPatch
import ipywidgets as widgets
from IPython.display import display
import yaml
import cv2

# Disable pyplot's implicit auto-display to avoid duplicate figure rendering.
plt.ioff()

# --- Load map ---
MAP_PATH = os.path.abspath(MAP_PATH_)


map_img = plt.imread(MAP_PATH)
h, w = map_img.shape[:2]

# Grayscale map used for path planning (A*)
planner_img = cv2.imread(MAP_PATH, cv2.IMREAD_GRAYSCALE)
planner_img = (map_img * 255).astype(np.uint8)

fig, ax = plt.subplots(figsize=(10, 10))
ax.imshow(map_img, origin="lower", cmap="gray")
ax.set_xlim(0, w)
ax.set_ylim(0, h)

# --- State ---
state = {
    "mode": "WAIT_START",
    "dragging": False,
    "start_point": None,
    "preview_arrow": None,
    "pose_pairs": [],
    "current_index": None
}

TASK_FILE_PATH = os.path.dirname(MAP_PATH) + "/navpoints.yaml"
if os.path.exists(TASK_FILE_PATH):
    with open(TASK_FILE_PATH, "r") as f:
        loaded = yaml.safe_load(f)
        if loaded and "pairs" in loaded:
            state["pose_pairs"] = loaded["pairs"]
            state["current_index"] = len(state["pose_pairs"]) - 1

drawn_artists = []

# --- Utilities ---
def compute_theta(x0, y0, x1, y1):
    theta_img = np.arctan2(y1 - y0, x1 - x0)
    theta_inv = np.pi - theta_img
    theta_inv = (theta_inv + np.pi) % (2 * np.pi) - np.pi
    return float(theta_inv)

def draw_arrow(x0, y0, x1, y1, color, linewidth=2):
    dx = x1 - x0
    dy = y1 - y0
    x1 = x0 - dx
    y1 = y0 + dy
    arrow = FancyArrowPatch(
        (x0, y0), (x1, y1),
        arrowstyle='->',
        mutation_scale=15,
        linewidth=linewidth,
        color=color
    )
    ax.add_patch(arrow)
    return arrow

def compute_path_for_pair(pair):
    if pair["goal"] is None:
        return None

    s = pair["start"]
    g = pair["goal"]

    # A* expects (row, col) ~= (y, x)
    start_rc = (
        int(np.clip(round(s["y"]), 0, h - 1)),
        int(np.clip(round(s["x"]), 0, w - 1))
    )
    goal_rc = (
        int(np.clip(round(g["y"]), 0, h - 1)),
        int(np.clip(round(g["x"]), 0, w - 1))
    )

    try:
        return get_astar_path(planner_img, start_rc, goal_rc)
    except Exception as e:
        print(f"A* failed for start={start_rc}, goal={goal_rc}: {e}")
        return None

def redraw():
    for a in drawn_artists:
        a.remove()
    drawn_artists.clear()

    idx = state["current_index"]
    if idx is None:
        fig.canvas.draw_idle()
        return

    pair = state["pose_pairs"][idx]

    s = pair["start"]
    drawn_artists.append(
        draw_arrow(s["x"], s["y"],
                   s["x"] + np.cos(s["theta"]) * 20,
                   s["y"] + np.sin(s["theta"]) * 20,
                   "green", 3)
    )

    if pair["goal"] is not None:
        g = pair["goal"]
        drawn_artists.append(
            draw_arrow(g["x"], g["y"],
                    g["x"] + np.cos(g["theta"]) * 20,
                    g["y"] + np.sin(g["theta"]) * 20,
                    "blue", 3)
        )

        # Draw A* path when both start and goal exist
        path = compute_path_for_pair(pair)
        if path is not None and len(path) > 0:
            path_np = np.asarray(path)
            (line,) = ax.plot(path_np[:, 1], path_np[:, 0], 'r-', linewidth=2, alpha=0.6)
            drawn_artists.append(line)

    fig.canvas.draw_idle()

def update_dropdown():
    pair_dropdown.options = [
        (f"Pair {i}", i) for i in range(len(state["pose_pairs"]))
    ]
    if state["pose_pairs"]:
        state["current_index"] = len(state["pose_pairs"]) - 1
        pair_dropdown.value = state["current_index"]
    else:
        state["current_index"] = None

def update_status():
    if state["mode"] == "WAIT_START":
        status_label.value = "<b>Mode:</b> Define START pose (green)"
    else:
        status_label.value = "<b>Mode:</b> Define GOAL pose (blue)"

# --- Mouse Events ---
def on_press(event):
    if event.inaxes != ax:
        return
    state["dragging"] = True
    state["start_point"] = (event.xdata, event.ydata)

def on_motion(event):
    if not state["dragging"] or event.inaxes != ax:
        return

    x0, y0 = state["start_point"]
    x1, y1 = event.xdata, event.ydata

    if state["preview_arrow"]:
        state["preview_arrow"].remove()

    state["preview_arrow"] = FancyArrowPatch(
        (x0, y0), (x1, y1),
        arrowstyle='->',
        linestyle='dashed',
        color='red'
    )
    ax.add_patch(state["preview_arrow"])
    fig.canvas.draw_idle()

def on_release(event):
    if not state["dragging"] or event.inaxes != ax:
        return

    x0, y0 = state["start_point"]
    x1, y1 = event.xdata, event.ydata
    theta = compute_theta(x0, y0, x1, y1)

    pose = {
        "x": round(x0),
        "y": round(y0),
        "theta": float(theta)
    }

    if state["mode"] == "WAIT_START":
        # draw start arrow immediately
        state["pose_pairs"].append({
            "start": pose,
            "goal": None
        })
        state["current_index"] = len(state["pose_pairs"]) - 1

        state["mode"] = "WAIT_GOAL"

    else:
        idx = state["current_index"]
        state["pose_pairs"][idx]["goal"] = pose
        state["mode"] = "WAIT_START"
        update_dropdown()

    if state["preview_arrow"]:
        state["preview_arrow"].remove()
        state["preview_arrow"] = None

    state["dragging"] = False
    update_status()
    redraw()

fig.canvas.mpl_connect("button_press_event", on_press)
fig.canvas.mpl_connect("motion_notify_event", on_motion)
fig.canvas.mpl_connect("button_release_event", on_release)

# --- Widgets ---
prev_btn = widgets.Button(description="⬅ Prev")
next_btn = widgets.Button(description="Next ➡")
delete_btn = widgets.Button(description="Delete")
clear_btn = widgets.Button(description="Clear All")
save_btn = widgets.Button(description="Save YAML")
pair_dropdown = widgets.Dropdown(description="Pairs:")
status_label = widgets.HTML()

def on_prev(b):
    if state["pose_pairs"] and state["current_index"] > 0:
        state["current_index"] -= 1
        pair_dropdown.value = state["current_index"]
        redraw()

def on_next(b):
    if state["pose_pairs"] and state["current_index"] < len(state["pose_pairs"]) - 1:
        state["current_index"] += 1
        pair_dropdown.value = state["current_index"]
        print(state["pose_pairs"] )
        redraw()

def on_delete(b):
    idx = state["current_index"]
    if idx is not None:
        state["pose_pairs"].pop(idx)
        update_dropdown()
        redraw()

def on_clear(b):
    state["pose_pairs"].clear()
    update_dropdown()
    redraw()

def on_save(b):
    with open(TASK_FILE_PATH, "w") as f:
        yaml.dump({"pairs": state["pose_pairs"]}, f)

def on_dropdown_change(change):
    state["current_index"] = change["new"]
    redraw()

prev_btn.on_click(on_prev)
next_btn.on_click(on_next)
delete_btn.on_click(on_delete)
clear_btn.on_click(on_clear)
save_btn.on_click(on_save)
pair_dropdown.observe(on_dropdown_change, names="value")

# Instructions
- Click-drag-release to create a pose (rviz-stile)
- The first pose inserted is the start, the next one is the goal.
- Insert as many poses as you want, review/delete them with arrows or the drop-down widget
- When finished, click on `Save YAML`, this will create a `navpoints.yaml` file in the map folder
- These waypoints will be used for evaluation/training

In [ ]:
update_status()
update_dropdown()
redraw()
display(widgets.VBox([
    widgets.HBox([prev_btn, next_btn, delete_btn, clear_btn, save_btn]),
    pair_dropdown,
    status_label,
    fig.canvas,
]))